# SECOM Industrial Fault Detection - Colab Training Notebook

Use this notebook in Google Colab to train the machine learning models and export the Joblib artifact required by the Streamlit Cloud app.

Output artifact:

`models/secom_fault_detection_model.joblib`

## 1. Clone Your GitHub Repository

Replace `REPO_URL` with your own GitHub repository URL after you upload the project files.

In [ ]:
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/disharaHettiarachchi/SECOM.git"
BRANCH = "main"
PROJECT_DIR = "/content/secom-project"

if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)

subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, PROJECT_DIR], check=True)
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)

print("Project directory:", os.getcwd())

## 2. Install Requirements

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)

## 3. Load and Check the Dataset

In [ ]:
from src.data_loader import dataset_summary, load_secom_dataset, split_features_and_target
from src.preprocessing import build_feature_quality_report

data = load_secom_dataset()
features, target = split_features_and_target(data)
summary = dataset_summary(data)
quality_report, quality_counts = build_feature_quality_report(features)

print("Records:", summary["records"])
print("Sensor features:", summary["features"])
print("Normal/pass records:", summary["normal_records"])
print("Fault/fail records:", summary["fault_records"])
print("Feature quality counts:", quality_counts)

## 4. Train and Compare Models

The default setting uses scikit-learn models only so the exported model is simple to deploy on Streamlit Cloud. Model selection prioritises fault-class F1 and recall rather than accuracy alone. Existing cell outputs may belong to an older run, so use Runtime > Restart session and run all before exporting the final artifact. If you later want to experiment with XGBoost, set `include_xgboost=True` and add `xgboost` to `requirements.txt` before deployment.

In [ ]:
from src.model_training import TrainingConfig, train_and_evaluate_models

config = TrainingConfig(
    test_size=0.25,
    random_state=42,
    missing_threshold=0.50,
    selected_features=80,
    include_xgboost=False,
    selection_metric="f1",
    decision_threshold=0.50,
)

training_result = train_and_evaluate_models(features, target, config)
comparison = training_result["comparison_table"]
comparison

## 5. Save the Best Model Artifact

The artifact includes the preprocessing pipeline, best model, model metrics, model comparison table, and metadata needed by the Streamlit app.

In [ ]:
from pathlib import Path
from src.model_training import create_model_artifact, save_model_artifact

artifact = create_model_artifact(training_result)
artifact_path = save_model_artifact(artifact)

Path("data/processed").mkdir(parents=True, exist_ok=True)
comparison.to_csv("data/processed/model_comparison_results.csv", index=False)

print("Best model:", artifact["model_name"])
print("Saved artifact:", artifact_path)
print("Saved comparison CSV: data/processed/model_comparison_results.csv")

## 6. Download Files

Download the Joblib file and place it in your GitHub repository under `models/`. Then commit and push it before deploying on Streamlit Cloud.

In [ ]:
from google.colab import files

files.download("models/secom_fault_detection_model.joblib")
files.download("data/processed/model_comparison_results.csv")

## 7. Optional Git Commands

If you prefer to push from your own computer, skip this section. After downloading the model, copy it into the local project folder and run:

```bash
git add models/secom_fault_detection_model.joblib data/processed/model_comparison_results.csv
git commit -m "Add trained SECOM model artifact"
git push
```